## Logistic Regression

In [25]:
from src import LogisticRegression
from src import train_test_split, standardize, evaluate_model, handle_missing_values
import pandas as pd
import numpy as np

# Load data
print("Loading data...")
train_df = pd.read_csv('data/train.csv')
print(f"Dataset shape: {train_df.shape}")

# Separate features and target
y = train_df['is_fraud'].values
X = train_df.drop(['is_fraud', 'ID'], axis=1, errors='ignore')

# PENTING: Ambil HANYA kolom numeric
print(f"\nOriginal columns: {X.shape[1]}")
X_numeric = X.select_dtypes(include=[np.number])
print(f"Numeric columns: {X_numeric.shape[1]}")

# Convert to numpy array
X_values = X_numeric.values

# Handle missing values (NaN)
print("\nHandling missing values...")
X_clean = handle_missing_values(X_values, strategy='mean')

# Standardize (sekarang aman karena semua numeric)
print("Standardizing features...")
X_standardized = standardize(X_clean)

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X_standardized, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Features: {X_train.shape[1]}")


Loading data...
Dataset shape: (100000, 31)

Original columns: 29
Numeric columns: 22

Handling missing values...
Standardizing features...

Training samples: 80000
Validation samples: 20000
Features: 22


In [26]:
# Train
print("\n" + "="*70)
print("TRAINING LOGISTIC REGRESSION")
print("="*70)

model = LogisticRegression(
    learning_rate=0.1,
    n_iterations=1000,
    regularization='l2',
    lambda_=0.01,
    verbose=True
)

model.fit(X_train, y_train)


TRAINING LOGISTIC REGRESSION
Iteration 0/1000 - Cost: 0.6801
Iteration 100/1000 - Cost: 0.4098
Iteration 200/1000 - Cost: 0.4011
Iteration 300/1000 - Cost: 0.4003
Iteration 400/1000 - Cost: 0.4002
Iteration 500/1000 - Cost: 0.4002
Iteration 600/1000 - Cost: 0.4002
Iteration 700/1000 - Cost: 0.4002
Iteration 800/1000 - Cost: 0.4002
Iteration 900/1000 - Cost: 0.4002
Training completed! Final cost: 0.4002


LogisticRegression(learning_rate=0.1, n_iterations=1000, regularization=l2, lambda_=0.01)

In [27]:
# Evaluate
print("\n" + "="*70)
print("EVALUATION")
print("="*70)

y_pred = model.predict(X_val)
metrics = evaluate_model(y_val, y_pred)

print(f"\nAccuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1_score']:.4f}")
print(f"\nConfusion Matrix:\n{metrics['confusion_matrix']}")


EVALUATION

Accuracy:  0.8599
Precision: 0.8043
Recall:    0.8599
F1-Score:  0.8312

Confusion Matrix:
[[17192     7]
 [ 2795     6]]


In [ ]:
# Save
print("\n" + "="*70)
print("SAVING MODEL")
print("="*70)

model.save_model('models/savedmodels/logistic_regression.pkl')
print("✓ Model saved successfully!")


SAVING MODEL
✓ Model saved successfully!


In [33]:
from src.logistic_regression import LogisticRegression

# Load
model = LogisticRegression.load_model('models/savedmodels/logistic_regression.pkl')

# Get info (untuk Logistic Regression)
params = model.get_params()

print(f"Learning Rate: {params['learning_rate']}")
print(f"Iterations: {params['n_iterations']}")
print(f"Regularization: {params['regularization']}")
print(f"Lambda: {params['lambda_']}")
print(f"Number of Features: {len(params['weights'])}")
print(f"Bias: {params['bias']:.4f}")
print(f"Final Cost: {params['cost_history'][-1]:.4f}")


Learning Rate: 0.1
Iterations: 1000
Regularization: l2
Lambda: 0.01
Number of Features: 22
Bias: -1.8409
Final Cost: 0.4002


In [29]:
test_df = pd.read_csv('data/test.csv')
print(f"Test data shape: {test_df.shape}")

# Preprocessing (SAMA dengan training!)
X_test = test_df.drop(['ID'], axis=1, errors='ignore')
X_test_numeric = X_test.select_dtypes(include=[np.number])
print(f"Test numeric columns: {X_test_numeric.shape[1]}")

X_test_values = X_test_numeric.values
X_test_clean = handle_missing_values(X_test_values, strategy='mean')
X_test_standardized = standardize(X_test_clean)

print(f"Test data processed: {X_test_standardized.shape}")

# Predict
print("\nMaking predictions...")
test_predictions = model.predict(X_test_standardized)
test_probabilities = model.predict_proba(X_test_standardized)

print(f"✓ Predictions completed!")
print(f"  Total: {len(test_predictions)}")
print(f"  Fraud: {np.sum(test_predictions == 1)} ({np.sum(test_predictions == 1)/len(test_predictions)*100:.2f}%)")
print(f"  Not Fraud: {np.sum(test_predictions == 0)} ({np.sum(test_predictions == 0)/len(test_predictions)*100:.2f}%)")

Test data shape: (100000, 30)
Test numeric columns: 22
Test data processed: (100000, 22)

Making predictions...
✓ Predictions completed!
  Total: 100000
  Fraud: 35 (0.03%)
  Not Fraud: 99965 (99.97%)


In [30]:
# Create submission
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'is_fraud': test_predictions
})

print("\nSubmission preview:")
print(submission.head(10))
print(f"\nShape: {submission.shape}")

# Save
submission.to_csv('data/submission.csv', index=False)
print("\n✓ Submission saved: submission.csv")
print("📤 Ready to upload to Kaggle!")

print("\n" + "="*70)
print("✅ ALL TASKS COMPLETED!")
print("="*70)


Submission preview:
   ID  is_fraud
0   0         0
1   1         0
2   2         0
3   3         0
4   4         0
5   5         0
6   6         0
7   7         0
8   8         0
9   9         0

Shape: (100000, 2)

✓ Submission saved: submission.csv
📤 Ready to upload to Kaggle!

✅ ALL TASKS COMPLETED!


In [ ]:
from src.logistic_regression import LogisticRegression
from src import train_test_split, standardize, handle_missing_values
import pandas as pd
import numpy as np

# Load model yang sudah ada
print("Loading existing model...")
model = LogisticRegression.load_model('models/savedmodels/logistic_regression_retrained.pkl')
print(f"Old parameters: LR={model.learning_rate}, Iter={model.n_iterations}")

# Load data baru
train_df = pd.read_csv('data/train.csv')
y = train_df['is_fraud'].values
X = train_df.drop(['is_fraud', 'ID'], axis=1, errors='ignore')
X_numeric = X.select_dtypes(include=[np.number]).values
X_clean = handle_missing_values(X_numeric, strategy='mean')
X_std = standardize(X_clean)
X_train, X_val, y_train, y_val = train_test_split(X_std, y, test_size=0.2)

# Train ulang (ini akan RESET model!)
print("\nRetraining model...")
model.fit(X_train, y_train)

# Evaluate
accuracy = model.score(X_val, y_val)
print(f"New accuracy: {accuracy:.4f}")

# Save (overwrite atau save dengan nama baru)
model.save_model('models/savedmodels/logistic_regression_retrained.pkl')
print("✓ Retrained model saved!")


Loading existing model...
Old parameters: LR=0.1, Iter=1000

Retraining model...
Iteration 0/1000 - Cost: 0.6802
Iteration 100/1000 - Cost: 0.4099
Iteration 200/1000 - Cost: 0.4011
Iteration 300/1000 - Cost: 0.4004
Iteration 400/1000 - Cost: 0.4003
Iteration 500/1000 - Cost: 0.4003
Iteration 600/1000 - Cost: 0.4003
Iteration 700/1000 - Cost: 0.4003
Iteration 800/1000 - Cost: 0.4003
Iteration 900/1000 - Cost: 0.4003
Training completed! Final cost: 0.4003
New accuracy: 0.8602
✓ Retrained model saved!
